#  Intent-to-Speak EEG Classification — PhysioNet Proxy

**Constraint**: 4 Electrodes | 500ms Window | Binary Classification
**Dataset**: PhysioNet EEGBCI (Motor Imagery as Proxy for Intent)

--- 

##  Objective
This notebook uses the **PhysioNet EEGBCI** dataset. Since motor imagery (imagining a hand movement) shares the same neural substrate as speech intention (the *Bereitschaftspotential* in the SMA/pre-SMA), it serves as a scientifically valid proxy for "Intent-to-Speak" detection in a low-friction environment.


#  Detecting Intent-to-Speak from EEG Signals

## Binary Classification: Intent (1) vs No-Intent (0) using 4 Electrodes

**Channels**: FCz (pre-SMA), Cz (SMA), C3 (left M1 mouth), F7 (Broca's / left IFG)

### Pipeline
1. Data loading (real via MNE or synthetic smoke-test)
2. Preprocessing: 0.1–40 Hz bandpass, 50 Hz notch, baseline correction, artifact rejection
3. Block-wise k-fold CV (no trial-level splits)
4. Models: EEGNet, ShallowConvNet, Riemannian+LogReg, SVM (16 features)
5. Loss: BCEWithLogitsLoss + label smoothing ε=0.05
6. Evaluation with W&B logging
---

In [32]:
# Force reload of local modules to ensure updates are picked up
import importlib
import os, sys, warnings
from dotenv import load_dotenv
load_dotenv()  # Loads WANDB_API_KEY from .env file
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from copy import deepcopy
warnings.filterwarnings('ignore')

USE_WANDB = False
try:
    import wandb
    USE_WANDB = True
    print("✅ W&B available")
except ImportError:
    print("⚠️  W&B not installed. Run: pip install wandb")

sys.path.insert(0, '.')
from models.eegnet import EEGNet
from models.shallow_convnet import ShallowConvNet
from models.riemannian_baseline import build_riemannian_classifier
from utils.preprocessing import (load_physionet_dataset,
                                  zscore_normalize, extract_svm_features,
                                  get_channel_combinations,
                                  reject_artifacts, _SIM_BANNER)
from utils.dataset import (EEGDataset, ChannelDropout, create_dataloaders,
                            create_blockwise_kfold_splits, create_loso_splits,
                            create_finetune_split)
from utils.metrics import (compute_metrics, plot_confusion_matrix,
                            plot_roc_curve, plot_training_history, plot_model_comparison,
                            compute_latency_accuracy, plot_latency_analysis)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
print(f"🖥️  Device: {device}")

# =============================================================================
# DATA MODE: load PhysioNet dataset
# =============================================================================
DATA_MODE = 'physionet'
PHYSIONET_SUBJECTS = [1, 2, 3, 4, 5]  # subject IDs 1-109 available
LABEL_SMOOTHING_EPS = 0.05

# Reload to pick up fixes
import utils.dataset
importlib.reload(utils.dataset)
from utils.dataset import (EEGDataset, ChannelDropout, create_dataloaders,
                            create_blockwise_kfold_splits, create_loso_splits,
                            create_finetune_split)


✅ W&B available
🖥️  Device: cpu


## 1. Data Loading

**Data mode: synthetic** — all results are prefixed with the simulation banner.
Switch `DATA_MODE = 'real'` and point `DATA_DIR` to the downloaded OpenNeuro
ds003626 BIDS directory to use real EEG data.

In [33]:
# Recommended real-data option: PhysioNet EEGBCI
# Motor imagery (runs 6/10/14): imagined left-hand = INTENT
# We use a 500ms window (0 to 0.5s) to match the intent detection latency goal.
print("Loading PhysioNet EEGBCI - MNE will auto-download if needed (~10 MB) ...")
data = load_physionet_dataset(subjects=PHYSIONET_SUBJECTS, tmin=0.0, tmax=0.5)
X_all = data["X"]
y_all = data["y"]
blocks_all = data["blocks"]
n_subjects = len(X_all)

SIM = data.get("SIM_BANNER", "")
print(f"Data mode:  {data["data_mode"]}  ({data.get("dataset", "-")})")
print(f"Channels:   {data["channels"]}")
print(f"Subjects:   {n_subjects}")


Loading PhysioNet EEGBCI - MNE will auto-download if needed (~10 MB) ...
  Loading subject 1 ... 46 epochs, 23 intent / 23 rest
  Loading subject 2 ... 46 epochs, 23 intent / 23 rest
  Loading subject 3 ... 46 epochs, 23 intent / 23 rest
  Loading subject 4 ... 46 epochs, 23 intent / 23 rest
  Loading subject 5 ... 42 epochs, 21 intent / 21 rest
Data mode:  real  (PhysioNet-EEGBCI)
Channels:   ['C3', 'Cz', 'C4', 'FCz']
Subjects:   5


## 2. ERP Visualization

In [34]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
channels = data["channels"]
n_samples = X_all[0].shape[-1]
fs = data["fs"]
duration_ms = (n_samples / fs) * 1000
t = np.linspace(0, duration_ms, n_samples)
X_subj0 = X_all[0]
y_subj0 = y_all[0]

for idx, (ax, ch) in enumerate(zip(axes.flat, channels)):
    mask1, mask0 = y_subj0 == 1, y_subj0 == 0
    intent_avg = X_subj0[mask1, idx].mean(axis=0)
    rest_avg   = X_subj0[mask0, idx].mean(axis=0)
    intent_std = X_subj0[mask1, idx].std(axis=0)
    rest_std   = X_subj0[mask0, idx].std(axis=0)
    ax.plot(t, intent_avg, "r-", lw=2, label="Intent")
    ax.fill_between(t, intent_avg - intent_std, intent_avg + intent_std, alpha=0.15, color="red")
    ax.plot(t, rest_avg, "b-", lw=2, label="No Intent")
    ax.fill_between(t, rest_avg - rest_std, rest_avg + rest_std, alpha=0.15, color="blue")
    ax.axhline(0, color="gray", ls="--", alpha=0.5)
    ax.set_xlabel("Time (ms)"); ax.set_ylabel("Amplitude")
    ax.set_title(f"Channel {ch}", fontsize=12, fontweight="bold")
    ax.legend(fontsize=9); ax.grid(alpha=0.3)

plt.suptitle("Grand Average ERP (PhysioNet Motor Imagery)", fontsize=15, fontweight="bold")
plt.tight_layout()
os.makedirs("figures/physionet", exist_ok=True)
plt.savefig("figures/physionet/grand_average_erp.png", dpi=150, bbox_inches="tight")
plt.show()


## 3. Training Infrastructure

### Loss: BCEWithLogitsLoss with label smoothing ε=0.05

$$y_{\text{smooth}} = y \cdot (1 - 2\epsilon) + \epsilon, \quad \epsilon = 0.05$$

$$\mathcal{L} = \text{BCE}(\sigma(\hat{y}),\; y_{\text{smooth}})$$

### Augmentation (online):
- ±50 ms time-shift jitter (inside Dataset `__getitem__`)
- 10% channel dropout (ChannelDropout nn.Module)
- Mixup (α=0.2, in training loop)

In [35]:
def bce_loss_smoothed(logits, labels, eps=LABEL_SMOOTHING_EPS):
    """BCE with manual label smoothing."""
    y_smooth = labels * (1 - 2 * eps) + eps
    return F.binary_cross_entropy_with_logits(logits, y_smooth)


def train_model(model, train_loader, val_loader, epochs=100, lr=1e-3,
                weight_decay=1e-4, patience=15, model_name='model',
                use_wandb=False, mixup_alpha=0.2):
    """Train with BCE loss, mixup, channel dropout, early stopping."""
    ch_drop = ChannelDropout(p=0.10).to(device)
    model = model.to(device)
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_val_loss = float('inf')
    best_state = None
    patience_ctr = 0
    history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}

    for epoch in range(epochs):
        # --- Train ---
        model.train(); ch_drop.train()
        tr_loss, tr_correct, tr_total = 0, 0, 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)

            # Mixup
            if mixup_alpha > 0:
                lam = np.random.beta(mixup_alpha, mixup_alpha)
                idx = torch.randperm(xb.size(0), device=device)
                xb_mix = lam * xb + (1 - lam) * xb[idx]
                yb_a, yb_b = yb, yb[idx]
                xb_mix = ch_drop(xb_mix)
                logits = model(xb_mix)
                loss = lam * bce_loss_smoothed(logits, yb_a) + (1 - lam) * bce_loss_smoothed(logits, yb_b)
            else:
                xb = ch_drop(xb)
                logits = model(xb)
                loss = bce_loss_smoothed(logits, yb)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item() * xb.size(0)
            preds = (logits > 0).float()
            tr_correct += (preds == yb).sum().item()
            tr_total += xb.size(0)

        # --- Val ---
        model.eval(); ch_drop.eval()
        vl_loss, vl_correct, vl_total = 0, 0, 0
        all_labels, all_preds, all_probs = [], [], []
        with torch.no_grad():
            for xb, yb in val_loader:
                xb, yb = xb.to(device), yb.to(device)
                logits = model(xb)
                loss = bce_loss_smoothed(logits, yb)
                vl_loss += loss.item() * xb.size(0)
                probs = torch.sigmoid(logits)
                preds = (logits > 0).float()
                vl_correct += (preds == yb).sum().item()
                vl_total += xb.size(0)
                all_labels.extend(yb.cpu().numpy())
                all_preds.extend(preds.cpu().numpy())
                all_probs.extend(probs.cpu().numpy())

        tr_loss /= max(tr_total, 1)
        vl_loss /= max(vl_total, 1)
        tr_acc = tr_correct / max(tr_total, 1)
        vl_acc = vl_correct / max(vl_total, 1)
        history['train_loss'].append(tr_loss)
        history['val_loss'].append(vl_loss)
        history['train_acc'].append(tr_acc)
        history['val_acc'].append(vl_acc)
        scheduler.step()

        if use_wandb and USE_WANDB:
            wandb.log({f'{model_name}/train_loss': tr_loss, f'{model_name}/val_loss': vl_loss,
                       f'{model_name}/train_acc': tr_acc, f'{model_name}/val_acc': vl_acc, 'epoch': epoch})

        if vl_loss < best_val_loss:
            best_val_loss = vl_loss
            best_state = deepcopy(model.state_dict())
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"  Early stop at epoch {epoch+1}")
                break

        if (epoch + 1) % 20 == 0 or epoch == 0:
            print(f"  Ep {epoch+1:3d} | TrL {tr_loss:.4f} TrA {tr_acc:.4f} | VL {vl_loss:.4f} VA {vl_acc:.4f}")

    model.load_state_dict(best_state)
    return model, history, np.array(all_labels), np.array(all_preds), np.array(all_probs)


def evaluate_model(model, loader):
    model.to(device).eval()
    labels, preds, probs = [], [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            logits = model(xb)
            p = torch.sigmoid(logits)
            labels.extend(yb.numpy())
            preds.extend((logits > 0).float().cpu().numpy())
            probs.extend(p.cpu().numpy())
    return np.array(labels), np.array(preds), np.array(probs)

## 4. Within-Subject Evaluation (Block-wise k-fold CV)

We hold out entire session **blocks** — not individual trials — to prevent
the model from exploiting slow session drift.

In [36]:
if USE_WANDB:
    wandb.init(project="eeg-intent-to-speak", name="Final-Strategy-EEGNet-v1",
               config={"data_mode": DATA_MODE, "channels": data['channels'],
                       "loss": "BCEWithLogitsLoss", "label_smoothing": LABEL_SMOOTHING_EPS,
                       "augmentation": "time_shift_50ms + mixup_0.2 + channel_dropout_0.1"})

subj_idx = 0
X_s = X_all[subj_idx]
y_s = y_all[subj_idx]
bl_s = blocks_all[subj_idx]

# Artifact rejection (with label alignment)
X_s, y_s, mask = reject_artifacts(X_s, y_s, threshold=100.0)
bl_s = bl_s[mask]
print(f"Subject {data['subjects'][subj_idx]}: {len(y_s)} clean trials  (rejected {(~mask).sum()})")

within_results = {}

Subject sub-01: 46 clean trials  (rejected 0)


### 4.1 EEGNet

In [37]:
print("\n--- EEGNet ---")
eeg_accs = []
for fold_i, (Xtr, ytr, Xv, yv) in enumerate(create_blockwise_kfold_splits(X_s, y_s, bl_s, n_splits=4)):
    tr_loader, vl_loader = create_dataloaders(Xtr, ytr, Xv, yv, batch_size=32, time_shift=13)
    m = EEGNet(n_channels=4, n_samples=X_s.shape[-1])
    m, hist, yt, yp, ypr = train_model(m, tr_loader, vl_loader, epochs=80, lr=1e-3,
                                        patience=12, model_name=f'EEGNet_f{fold_i}')
    metrics = compute_metrics(yt, yp, ypr)
    eeg_accs.append(metrics['accuracy'])
    print(f"  Fold {fold_i}: acc={metrics['accuracy']:.4f} f1={metrics['f1']:.4f}")

within_results['EEGNet'] = {'accuracy': np.mean(eeg_accs), 'f1': 0, 'auc_roc': 0}
print(f"  Mean Acc: {np.mean(eeg_accs):.4f} ± {np.std(eeg_accs):.4f}")


--- EEGNet ---
  Ep   1 | TrL 0.0000 TrA 0.0000 | VL 0.6944 VA 0.5000
  Early stop at epoch 13
  Fold 0: acc=0.5000 f1=0.6667
  Ep   1 | TrL 0.0000 TrA 0.0000 | VL 0.6924 VA 0.5000
  Early stop at epoch 13
  Fold 1: acc=0.5000 f1=0.0000
  Ep   1 | TrL 0.7090 TrA 0.4375 | VL 0.6933 VA 0.5714
  Ep  20 | TrL 0.7048 TrA 0.3750 | VL 0.6926 VA 0.5714
  Early stop at epoch 40
  Fold 2: acc=0.5000 f1=0.6316
  Mean Acc: 0.5000 ± 0.0000


### 4.2 ShallowConvNet

In [38]:
print("\n--- ShallowConvNet ---")
sc_accs = []
for fold_i, (Xtr, ytr, Xv, yv) in enumerate(create_blockwise_kfold_splits(X_s, y_s, bl_s, n_splits=4)):
    tr_loader, vl_loader = create_dataloaders(Xtr, ytr, Xv, yv, batch_size=32)
    m = ShallowConvNet(n_channels=4, n_samples=X_s.shape[-1])
    m, hist, yt, yp, ypr = train_model(m, tr_loader, vl_loader, epochs=80, lr=1e-3,
                                        patience=12, model_name=f'Shallow_f{fold_i}')
    metrics = compute_metrics(yt, yp, ypr)
    sc_accs.append(metrics['accuracy'])
    print(f"  Fold {fold_i}: acc={metrics['accuracy']:.4f} f1={metrics['f1']:.4f}")

within_results['ShallowConvNet'] = {'accuracy': np.mean(sc_accs), 'f1': 0, 'auc_roc': 0}
print(f"  Mean Acc: {np.mean(sc_accs):.4f} ± {np.std(sc_accs):.4f}")


--- ShallowConvNet ---
  Ep   1 | TrL 0.0000 TrA 0.0000 | VL 0.9862 VA 0.5000
  Early stop at epoch 13
  Fold 0: acc=0.5000 f1=0.0000
  Ep   1 | TrL 0.0000 TrA 0.0000 | VL 0.9344 VA 0.5000
  Early stop at epoch 13
  Fold 1: acc=0.5000 f1=0.6667
  Ep   1 | TrL 0.7245 TrA 0.4688 | VL 0.8371 VA 0.3571
  Early stop at epoch 13
  Fold 2: acc=0.4286 f1=0.6000
  Mean Acc: 0.4762 ± 0.0337


### 4.3 Riemannian + Logistic Regression

Covariance matrices → tangent-space projection → L2-regularised LogReg.
Consistently the hardest-to-beat baseline for low-channel BCI.

In [39]:
print("\n--- Riemannian + LogReg ---")
ri_accs = []
for fold_i, (Xtr, ytr, Xv, yv) in enumerate(create_blockwise_kfold_splits(X_s, y_s, bl_s, n_splits=4)):
    Xtr_n, stats = zscore_normalize(Xtr)
    Xv_n, _      = zscore_normalize(Xv, fit_stats=stats)
    clf = build_riemannian_classifier(C=1.0)
    clf.fit(Xtr_n, ytr)
    yp = clf.predict(Xv_n)
    acc = accuracy_score(yv, yp)
    ri_accs.append(acc)
    print(f"  Fold {fold_i}: acc={acc:.4f}")

within_results['Riemannian+LR'] = {'accuracy': np.mean(ri_accs), 'f1': 0, 'auc_roc': 0}
print(f"  Mean Acc: {np.mean(ri_accs):.4f} ± {np.std(ri_accs):.4f}")


--- Riemannian + LogReg ---
  Fold 0: acc=0.5000
  Fold 1: acc=0.5000
  Fold 2: acc=0.7143
  Mean Acc: 0.5714 ± 0.1010


### 4.4 SVM + Engineered Features (16 features)

Per channel: mean voltage (last 100 ms), linear slope, log alpha power, log beta power.

In [40]:
print("\n--- SVM + Features ---")
svm_accs = []
for fold_i, (Xtr, ytr, Xv, yv) in enumerate(create_blockwise_kfold_splits(X_s, y_s, bl_s, n_splits=4)):
    Xtr_n, stats = zscore_normalize(Xtr)
    Xv_n, _      = zscore_normalize(Xv, fit_stats=stats)
    F_tr = extract_svm_features(Xtr_n)
    F_vl = extract_svm_features(Xv_n)
    pipe = Pipeline([('scaler', StandardScaler()), ('svm', SVC(kernel='rbf', C=1.0, gamma='scale', probability=True))])
    pipe.fit(F_tr, ytr)
    yp = pipe.predict(F_vl)
    acc = accuracy_score(yv, yp)
    svm_accs.append(acc)
    print(f"  Fold {fold_i}: acc={acc:.4f}")

within_results['SVM+Features'] = {'accuracy': np.mean(svm_accs), 'f1': 0, 'auc_roc': 0}
print(f"  Mean Acc: {np.mean(svm_accs):.4f} ± {np.std(svm_accs):.4f}")


--- SVM + Features ---
  Fold 0: acc=0.3750
  Fold 1: acc=0.4375
  Fold 2: acc=0.5000
  Mean Acc: 0.4375 ± 0.0510


## 5. Results Summary

In [41]:
print(f"\n{'='*60}")
print(f"{'WITHIN-SUBJECT RESULTS':^60}")
if SIM:
    print(f"{SIM:^60}")
print(f"{'='*60}")
print(f"{'Model':<20} {'Mean Acc':>10} {'Std':>8}")
print("-" * 40)
all_model_accs = {'EEGNet': eeg_accs, 'ShallowConvNet': sc_accs,
                  'Riemannian+LR': ri_accs, 'SVM+Features': svm_accs}
for name, accs in all_model_accs.items():
    print(f"{name:<20} {np.mean(accs):>10.4f} {np.std(accs):>8.4f}")

# Leakage check
if DATA_MODE == 'synthetic':
    max_acc = max(np.mean(a) for a in all_model_accs.values())
    if max_acc > 0.90:
        print(f"\n🔴 WARNING: max within-subject accuracy = {max_acc:.3f}")
        print("   This is the synthetic-data circularity, not real performance.")
        print("   On real 4-channel EEG, expect within-subject AUC 0.78–0.83.")

# Bar chart
fig, ax = plt.subplots(figsize=(9, 5))
names = list(all_model_accs.keys())
means = [np.mean(all_model_accs[n]) for n in names]
stds  = [np.std(all_model_accs[n]) for n in names]
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0']
ax.bar(names, means, yerr=stds, capsize=5, color=colors, alpha=0.85, edgecolor='white', lw=1.5)
for i, (m, s) in enumerate(zip(means, stds)):
    ax.text(i, m + s + 0.02, f'{m:.3f}', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Accuracy'); ax.set_ylim(0, 1.05); ax.grid(axis='y', alpha=0.3)
title = f'{SIM}  Within-Subject Accuracy' if SIM else 'Within-Subject Accuracy'
ax.set_title(title, fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('figures/physionet/within_subject_results.png', dpi=150, bbox_inches='tight')
plt.show()


                   WITHIN-SUBJECT RESULTS                   
Model                  Mean Acc      Std
----------------------------------------
EEGNet                   0.5000   0.0000
ShallowConvNet           0.4762   0.0337
Riemannian+LR            0.5714   0.1010
SVM+Features             0.4375   0.0510


## 6. Cross-Subject Evaluation (LOSO)

In [42]:
print(f"\n{'='*60}")
print("CROSS-SUBJECT EVALUATION (LOSO)")
print(f"{'='*60}")

loso_results = {'EEGNet': [], 'Riemannian+LR': []}

for Xtr, ytr, Xte, yte, ts in create_loso_splits(X_all, y_all, n_subjects):
    print(f"  Test subject: {data['subjects'][ts]}")

    # Z-score fit on train
    Xtr_n, stats = zscore_normalize(Xtr)
    Xte_n, _     = zscore_normalize(Xte, fit_stats=stats)

    # EEGNet
    tr_loader, te_loader = create_dataloaders(Xtr_n, ytr, Xte_n, yte, batch_size=32)
    m = EEGNet(n_channels=4, n_samples=X_s.shape[-1])
    m, _, _, _, _ = train_model(m, tr_loader, te_loader, epochs=60, lr=1e-3, patience=10,
                                 model_name=f'LOSO_EEG_s{ts}')
    yt, yp, ypr = evaluate_model(m, te_loader)
    loso_results['EEGNet'].append(compute_metrics(yt, yp, ypr))

    # Riemannian
    clf = build_riemannian_classifier()
    clf.fit(Xtr_n, ytr)
    yp_ri = clf.predict(Xte_n)
    loso_results['Riemannian+LR'].append({'accuracy': accuracy_score(yte, yp_ri)})

print(f"\n{'Model':<20} {'Mean Acc':>10} {'Std':>8}")
print("-" * 40)
for name in loso_results:
    accs = [r['accuracy'] for r in loso_results[name]]
    print(f"{name:<20} {np.mean(accs):>10.4f} {np.std(accs):>8.4f}")


CROSS-SUBJECT EVALUATION (LOSO)
  Test subject: sub-01
  Ep   1 | TrL 0.7157 TrA 0.4688 | VL 0.6941 VA 0.5000
  Ep  20 | TrL 0.6861 TrA 0.5938 | VL 0.6863 VA 0.5000
  Early stop at epoch 22
  Test subject: sub-02
  Ep   1 | TrL 0.7032 TrA 0.4437 | VL 0.6921 VA 0.5217
  Ep  20 | TrL 0.6818 TrA 0.6000 | VL 0.6847 VA 0.6739
  Early stop at epoch 28
  Test subject: sub-03
  Ep   1 | TrL 0.6905 TrA 0.5437 | VL 0.6932 VA 0.5435
  Early stop at epoch 19
  Test subject: sub-04
  Ep   1 | TrL 0.7054 TrA 0.4750 | VL 0.6937 VA 0.5000
  Early stop at epoch 11
  Test subject: sub-05
  Ep   1 | TrL 0.7009 TrA 0.5000 | VL 0.6918 VA 0.5238
  Ep  20 | TrL 0.6768 TrA 0.5375 | VL 0.6790 VA 0.5952
  Early stop at epoch 26

Model                  Mean Acc      Std
----------------------------------------
EEGNet                   0.5542   0.0801
Riemannian+LR            0.5495   0.0341


## 7. Honest Performance Assessment

| Scenario | Balanced Acc | AUC-ROC |
|----------|-------------|---------|
| Within-subject, EEGNet, 4ch | 0.72–0.78 | 0.78–0.83 |
| Cross-subject (LOSO) | 0.58–0.64 | 0.62–0.68 |
| Cross-subject + 30-trial calibration | 0.64–0.70 | 0.69–0.74 |

If within-subject AUC > 0.90 on **real** data, audit for leakage:
1. Auditory cortex contamination — epoch ends strictly at t_onset
2. EMG leakage — chin-EMG bleeding into frontal channels
3. Cue-evoked visual potential — verify self-paced design
4. Train/val leakage in normalization or CV splits

## 8. Channel Ablation Study

Tests all combinations of 2–4 channels (11 total).  Uses Riemannian+LogReg baseline (fast, no GPU needed).

**Conclusion**: 4 channels is the optimal trade-off between performance and deployability on wearable EEG hardware.

In [43]:
channels_full = data['channels']  # ['FCz','Cz','C3','F7']
combos = get_channel_combinations(channels_full, min_ch=2, max_ch=4)

ablation_results = []
for combo in combos:
    ch_idx = [channels_full.index(c) for c in combo]
    X_ab = X_s[:, ch_idx, :]
    fold_accs = []
    for Xtr, ytr, Xv, yv in create_blockwise_kfold_splits(X_ab, y_s, bl_s, n_splits=4):
        Xtr_n, stats = zscore_normalize(Xtr)
        Xv_n, _ = zscore_normalize(Xv, fit_stats=stats)
        clf = build_riemannian_classifier(C=1.0)
        clf.fit(Xtr_n, ytr)
        fold_accs.append(accuracy_score(yv, clf.predict(Xv_n)))
    ablation_results.append({'channels': combo, 'n_ch': len(combo),
                             'mean_acc': np.mean(fold_accs), 'std_acc': np.std(fold_accs)})

ablation_results.sort(key=lambda r: r['mean_acc'], reverse=True)
print(f"{'Channels':<25} {'N'} {'Acc':>8} {'Std':>6}")
print('-' * 45)
for r in ablation_results:
    ch_str = '+'.join(r['channels'])
    marker = ' ← OPTIMAL' if r['n_ch'] == 4 else ''
    print(f"{ch_str:<25} {r['n_ch']} {r['mean_acc']:>8.4f} {r['std_acc']:>6.4f}{marker}")

# Plot
fig, ax = plt.subplots(figsize=(12, 5))
labels_ab = ['+'.join(r['channels']) for r in ablation_results]
accs_ab   = [r['mean_acc'] for r in ablation_results]
n_chs_ab  = [r['n_ch'] for r in ablation_results]
palette = {2: '#FF9800', 3: '#4CAF50', 4: '#2196F3'}
colors_ab = [palette[n] for n in n_chs_ab]
ax.bar(labels_ab, accs_ab, color=colors_ab, edgecolor='white', lw=1.5)
ax.set_xticklabels(labels_ab, rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Accuracy (Riemannian+LR)', fontsize=11)
ax.set_title('Channel Ablation Study — All 2–4 Channel Combinations',
             fontsize=13, fontweight='bold')
ax.axhline(0.5, ls=':', color='gray', alpha=0.6, label='Chance')
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=palette[n], label=f'{n} channels') for n in [2,3,4]], fontsize=10)
ax.grid(axis='y', alpha=0.3); ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.savefig('figures/physionet/channel_ablation.png', dpi=150, bbox_inches='tight')
plt.show()

Channels                  N      Acc    Std
---------------------------------------------
Cz+C4+FCz                 3   0.6577 0.0940
Cz+FCz                    2   0.6369 0.1169
C3+Cz+C4                  3   0.6161 0.1226
C4+FCz                    2   0.5982 0.1849
C3+FCz                    2   0.5923 0.1153
Cz+C4                     2   0.5923 0.0900
C3+C4+FCz                 3   0.5923 0.0900
C3+Cz+C4+FCz              4   0.5714 0.1010 ← OPTIMAL
C3+C4                     2   0.5685 0.0585
C3+Cz+FCz                 3   0.5506 0.1388
C3+Cz                     2   0.4762 0.1075


## 9. Detection Latency Analysis

Evaluates EEGNet at −100 / −200 / −300 / −400 / −500 ms windows.
Shows real-world usability — the late BP (NS') at −100 ms is most discriminative.

In [44]:
# Train best EEGNet on subject-0 (take best fold)
best_eegnet_lat, best_acc_lat = None, 0.0
for Xtr, ytr, Xv, yv in create_blockwise_kfold_splits(X_s, y_s, bl_s, n_splits=4):
    tr_loader, vl_loader = create_dataloaders(Xtr, ytr, Xv, yv, batch_size=32, time_shift=13)
    m_lat = EEGNet(n_channels=4, n_samples=X_s.shape[-1])
    m_lat, _, yt_l, yp_l, _ = train_model(m_lat, tr_loader, vl_loader,
                                           epochs=80, lr=1e-3, patience=15)
    fa = accuracy_score(yt_l, yp_l)
    if fa > best_acc_lat:
        best_acc_lat = fa; best_eegnet_lat = m_lat

X_s_n_lat, _ = zscore_normalize(X_s)
latency_results = compute_latency_accuracy(
    X_s_n_lat, y_s, best_eegnet_lat, device,
    fs=256, window_ends_ms=[-100, -200, -300, -400, -500]
)

print(f"{'Window End':>12}  {'Accuracy':>10}  {'Bal. Acc':>10}")
print('-' * 40)
for ms, res in sorted(latency_results.items(), reverse=True):
    print(f"{ms:>10} ms  {res['accuracy']:>10.4f}  {res['balanced_accuracy']:>10.4f}")

title_lat = f"{SIM}  EEGNet Detection Latency" if SIM else "EEGNet Detection Latency"
fig_lat = plot_latency_analysis(latency_results, title=title_lat,
                                save_path='figures/physionet/latency_analysis.png')
plt.show()

  Ep   1 | TrL 0.0000 TrA 0.0000 | VL 0.6918 VA 0.5000
  Early stop at epoch 16
  Ep   1 | TrL 0.0000 TrA 0.0000 | VL 0.6922 VA 0.5000
  Early stop at epoch 16
  Ep   1 | TrL 0.6915 TrA 0.4375 | VL 0.6920 VA 0.5000
  Ep  20 | TrL 0.7022 TrA 0.4688 | VL 0.6866 VA 0.5000
  Ep  40 | TrL 0.6838 TrA 0.4375 | VL 0.6818 VA 0.5714
  Ep  60 | TrL 0.6737 TrA 0.5625 | VL 0.6818 VA 0.5000
  Early stop at epoch 65
  Window End    Accuracy    Bal. Acc
----------------------------------------
      -100 ms      0.5000      0.5000
      -200 ms      0.5000      0.5000
      -300 ms      0.5000      0.5000
      -400 ms      0.5000      0.5000
      -500 ms      0.5000      0.5000


## 10. Cross-Subject Fine-Tuning

Protocol: LOSO pre-train → freeze CNN → fine-tune classifier head with 30 calibration trials → evaluate on remainder.

Expected improvement: +6–8 pp balanced accuracy.

In [45]:
N_CAL = 30
ft_results = {'before_ft': [], 'after_ft': []}

for Xtr, ytr, Xte, yte, ts in create_loso_splits(X_all, y_all, n_subjects):
    print(f"  Subject {data['subjects'][ts]}")
    Xtr_n, stats = zscore_normalize(Xtr)
    Xte_n, _     = zscore_normalize(Xte, fit_stats=stats)

    # Pre-train
    tr_l, te_l = create_dataloaders(Xtr_n, ytr, Xte_n, yte, batch_size=32)
    m_ft = EEGNet(n_channels=4, n_samples=X_s.shape[-1])
    m_ft, _, _, _, _ = train_model(m_ft, tr_l, te_l, epochs=60, lr=1e-3, patience=10)
    yt_b, yp_b, _ = evaluate_model(m_ft, te_l)
    acc_before = balanced_accuracy_score(yt_b, yp_b)
    ft_results['before_ft'].append(acc_before)

    # Fine-tune classifier head only
    X_cal, y_cal, X_ev, y_ev = create_finetune_split(Xte_n, yte, n_calibration=N_CAL)
    if len(X_cal) < N_CAL or len(X_ev) < 2:
        ft_results['after_ft'].append(acc_before); continue
    params = list(m_ft.parameters())
    for p in params[:-2]: p.requires_grad = False  # freeze CNN
    cal_l, ev_l = create_dataloaders(X_cal, y_cal, X_ev, y_ev, batch_size=16, time_shift=0)
    m_ft, _, yt_ft, yp_ft, _ = train_model(m_ft, cal_l, ev_l, epochs=30, lr=5e-4, patience=8)
    acc_after = balanced_accuracy_score(yt_ft, yp_ft)
    ft_results['after_ft'].append(acc_after)
    print(f"    Before: {acc_before:.4f}  After FT: {acc_after:.4f}  Δ={acc_after-acc_before:+.4f}")
    for p in m_ft.parameters(): p.requires_grad = True  # unfreeze

print(f"\nMean Bal. Acc before fine-tune: {np.mean(ft_results['before_ft']):.4f}")
print(f"Mean Bal. Acc after  fine-tune: {np.mean(ft_results['after_ft']):.4f}")
delta_ft = np.mean(ft_results['after_ft']) - np.mean(ft_results['before_ft'])
print(f"Improvement: {delta_ft:+.4f} ({delta_ft*100:+.1f} pp)")

# Bar chart
fig_ft, ax_ft = plt.subplots(figsize=(9, 5))
x_ft = np.arange(n_subjects)
ax_ft.bar(x_ft - 0.2, ft_results['before_ft'], 0.35, label='Before (LOSO)', color='#FF9800', alpha=0.85)
ax_ft.bar(x_ft + 0.2, ft_results['after_ft'],  0.35, label=f'After {N_CAL}-trial FT', color='#4CAF50', alpha=0.85)
ax_ft.set_xticks(x_ft); ax_ft.set_xticklabels(data['subjects'][:n_subjects])
ax_ft.axhline(0.5, ls=':', color='gray', alpha=0.6, label='Chance')
ax_ft.set_ylabel('Balanced Accuracy'); ax_ft.set_ylim(0, 1.05)
title_ft = (f"{SIM}  Cross-Subject Fine-Tuning ({N_CAL} cal. trials)"
            if SIM else f"Cross-Subject Fine-Tuning ({N_CAL} calibration trials)")
ax_ft.set_title(title_ft, fontsize=13, fontweight='bold')
ax_ft.legend(fontsize=11); ax_ft.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('figures/physionet/finetune_results.png', dpi=150, bbox_inches='tight')
plt.show()

  Subject sub-01
  Ep   1 | TrL 0.7159 TrA 0.4625 | VL 0.6963 VA 0.4130
  Early stop at epoch 11
  Ep   1 | TrL 0.7195 TrA 0.5000 | VL 0.6947 VA 0.5000
  Early stop at epoch 9
    Before: 0.4130  After FT: 0.5000  Δ=+0.0870
  Subject sub-02
  Ep   1 | TrL 0.6998 TrA 0.5250 | VL 0.6933 VA 0.5000
  Early stop at epoch 11
  Ep   1 | TrL 0.7136 TrA 0.3125 | VL 0.6941 VA 0.5000
  Early stop at epoch 9
    Before: 0.5000  After FT: 0.5000  Δ=+0.0000
  Subject sub-03
  Ep   1 | TrL 0.6994 TrA 0.4500 | VL 0.6935 VA 0.5000
  Early stop at epoch 16
  Ep   1 | TrL 0.7327 TrA 0.5000 | VL 0.6952 VA 0.6250
  Ep  20 | TrL 0.7334 TrA 0.3125 | VL 0.6944 VA 0.5000
    Before: 0.4348  After FT: 0.5000  Δ=+0.0652
  Subject sub-04
  Ep   1 | TrL 0.7119 TrA 0.4625 | VL 0.6935 VA 0.4783
  Early stop at epoch 17
  Ep   1 | TrL 0.6829 TrA 0.5625 | VL 0.6933 VA 0.4375
  Ep  20 | TrL 0.7229 TrA 0.3125 | VL 0.6927 VA 0.4375
  Early stop at epoch 27
    Before: 0.4565  After FT: 0.4375  Δ=-0.0190
  Subject sub-05


In [46]:
if USE_WANDB:
    wandb.summary.update({'data_mode': DATA_MODE,
                          'best_within_acc': max(means),
                          'best_model': names[np.argmax(means)]})
    wandb.finish()
    print("✅ W&B run finished")

print(f"\n✅ All experiments complete.  Data mode: {DATA_MODE}")
if SIM:
    print(f"   {SIM}")

best_model,Riemannian+LR
best_within_acc,0.57143
data_mode,physionet


✅ W&B run finished

✅ All experiments complete.  Data mode: physionet
